In [15]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

project_root = Path.cwd().parent
sys.path.append(str(project_root / 'src'))

from ml_from_scratch.trees.decision_tree import DecisionTreeClassifier
from ml_from_scratch.preprocessing.scaler import MyStandardScaler

from ml_from_scratch.metrics.classification import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [16]:
data = load_breast_cancer()

X = data.data
y = data.target

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Classes:', np.unique(y))

X shape: (569, 30)
y shape: (569,)
Classes: [0 1]


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                    test_size = 0.2, random_state = 22, stratify = y)

In [18]:
scaler = MyStandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
model = DecisionTreeClassifier()

model.fit(X_train_scaled, y_train)

In [20]:
y_pred = model.predict(X_test_scaled).reshape(-1)

In [21]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1 Score : {f1:.4f}')

Accuracy : 0.8421
Precision: 0.9091
Recall   : 0.8333
F1 Score : 0.8696


In [22]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[36  6]
 [12 60]]


In [23]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
import time

sklearn_model = DecisionTreeClassifier(max_depth = 10, min_samples_split = 2, criterion = 'entropy', random_state = 22)

start = time.perf_counter()

sklearn_model.fit(X_train_scaled, y_train)

training_time = time.perf_counter() - start

sklearn_pred = sklearn_model.predict(X_test_scaled)

print(f'Training Time: {training_time:.6f}s')
print(f'Accuracy     : {accuracy_score(y_test, sklearn_pred):.4f}')
print(f'Precision    : {precision_score(y_test, sklearn_pred):.4f}')
print(f'Recall       : {recall_score(y_test, sklearn_pred):.4f}')
print(f'F1 Score     : {f1_score(y_test, sklearn_pred):.4f}')

Training Time: 0.017901s
Accuracy     : 0.8684
Precision    : 0.9254
Recall       : 0.8611
F1 Score     : 0.8921


In [24]:
custom_metrics = {
    
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred)
}

sklearn_metrics = {
    'Accuracy': accuracy_score(y_test, sklearn_pred),
    'Precision': precision_score(y_test, sklearn_pred),
    'Recall': recall_score(y_test, sklearn_pred),
    'F1': f1_score(y_test, sklearn_pred)
}

comparison = pd.DataFrame(
    {
        'My decision tree': custom_metrics,
        'Sklearn DecisionTreeClassifier': sklearn_metrics,
    }
)

comparison

,My decision tree,Sklearn DecisionTreeClassifier
Accuracy,0.842105,0.868421
Precision,0.909091,0.925373
Recall,0.833333,0.861111
F1,0.869565,0.892086


# Final Interpretation

The custom Decision Tree achieved solid classification performance on the Breast Cancer Wisconsin dataset.

On the test set, the model achieved 84.21% accuracy, 90.91% precision, 83.33% recall, and an F1 score of 86.96%. The confusion matrix was:

[[36  6]
 [12 60]]

This means that the model correctly classified 36 samples from class 0 and 60 samples from class 1, while producing 6 false positives and 12 false negatives. In this medical classification problem, the false negatives are particularly important because they represent positive cases that were incorrectly classified as negative.

The custom implementation was also compared with sklearn's DecisionTreeClassifier using the same general tree configuration. The sklearn model achieved 86.84% accuracy, 92.54% precision, 86.11% recall, and an F1 score of 89.21%.

The sklearn model therefore produced somewhat different results on this test split. This difference is expected because the custom implementation and sklearn use different internal strategies for finding and evaluating candidate splits, even when the main hyperparameters and entropy-based criterion are aligned. The comparison should therefore be considered a reference for validating the custom implementation rather than evidence that one implementation is universally better than the other.

Overall, this experiment demonstrated the complete workflow of implementing a Decision Tree classifier from scratch using NumPy, including entropy, information gain, split selection, recursive tree construction, stopping criteria, prediction, evaluation, and comparison with a standard machine learning library.